In [4]:
from pathlib import Path
import sys
import pandas as pd
import torch
import matplotlib.pyplot as plt
import xarray as xr
import yaml
import geopandas as gpd
import contextily as cx
from adjustText import adjust_text
import itertools


from neuralhydrology.nh_run import start_run, eval_run, finetune
from neuralhydrology.nh_run import continue_run
from neuralhydrology.utils.config import Config
from neuralhydrology.evaluation import get_tester, metrics

In [5]:
# -------- Paths ---------
CONFIG_PATH = Path("./1_basins_subset_test.yml")
RUNS_DIR = Path("runs")
ATTRIBUTES_FILE= Path('../../1_preparing_data/2_us_127_basins/data/attributes/attributes_other.csv')

In [6]:
# precip_products = [
#     # 'total_precipitation_sum'
#     # "chirps_precipitation",
#     "mswep_precipitation"
# ]

In [7]:
seeds = [888] #[111, 222, 333, 444, 555, 666, 777, 888] #  

base_non_precip_inputs = [
    "temperature_2m_max",
    "temperature_2m_min",
    "surface_net_solar_radiation_mean",
]

with open(CONFIG_PATH, "r") as f:
    base_config = yaml.safe_load(f)

use_gpu = torch.cuda.is_available() or torch.backends.mps.is_available()

# for r in range(1, len(precip_products) + 1):

precip_combos = [
    # ("camels_precipitation",),
    # ("total_precipitation_sum",),
    # ("chirps_precipitation",),
    # ("mswep_precipitation",),
    # ("chirps_precipitation", "mswep_precipitation"),
    # ("total_precipitation_sum", "chirps_precipitation"),
    ("total_precipitation_sum", "mswep_precipitation"),
    # ("total_precipitation_sum", "chirps_precipitation", "mswep_precipitation"),
]

# for precip_combo in itertools.combinations(precip_products, r):
for precip_combo in precip_combos:
    for seed in seeds:
        config = base_config.copy()

        config["dynamic_inputs"] = [*base_non_precip_inputs, *precip_combo]
        config["seed"] = seed

        precip_name = "_".join(precip_combo)
        config["experiment_name"] = (
            f"{precip_name}_seq_{config['seq_length']}"
            f"_{config['predict_last_n']}_epochs_{config['epochs']}"
            f"_hidden_{config['hidden_size']}"
            f"_dropout_{str(config['output_dropout']).replace('.', '')}"
            f"_fb_{config['initial_forget_bias']}"
            f"_seed{seed}"
        )

        temp_config_path = Path(f"temp_{precip_name}_seed{seed}.yml")
        with open(temp_config_path, "w") as f:
            yaml.dump(config, f)

        print(f"Running: {config['experiment_name']}")

        if use_gpu:
            start_run(config_file=temp_config_path)
        else:
            start_run(config_file=temp_config_path, gpu=-1)

        temp_config_path.unlink()  # clean up temp file after run

Running: total_precipitation_sum_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888
2026-05-20 16:02:37,116: Logging to /home/azureuser/NeuralHydrologyAzure/2_final_runs/2_us_127_basins/runs/total_precipitation_sum_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_2005_160237/output.log initialized.
2026-05-20 16:02:37,117: ### Folder structure created at /home/azureuser/NeuralHydrologyAzure/2_final_runs/2_us_127_basins/runs/total_precipitation_sum_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_2005_160237
2026-05-20 16:02:37,117: ### Run configurations for total_precipitation_sum_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888
2026-05-20 16:02:37,118: batch_size: 256
2026-05-20 16:02:37,118: clip_gradient_norm: 1
2026-05-20 16:02:37,118: data_dir: ../../1_preparing_data/2_us_127_basins/data
2026-05-20 16:02:37,119: dataset: generic
2026-05-20 16:02:37,119: device: cuda:0
2026-05-20 

2026-05-20 16:02:37,589: Loading basin data into xarray data set.
100%|██████████| 127/127 [00:01<00:00, 65.21it/s]
2026-05-20 16:02:39,601: Calculating target variable stds per basin
100%|██████████| 127/127 [00:00<00:00, 2111.12it/s]
2026-05-20 16:02:39,684: Create lookup table and convert to pytorch tensor
100%|██████████| 127/127 [00:02<00:00, 57.19it/s]
2026-05-20 16:02:41,979: Validation set to validate every 1 epoch(s), but 'validate_n_random_basins' not set or set to zero. Will validate on the entire validation set.
# Epoch 1: 100%|██████████| 1632/1632 [01:27<00:00, 18.73it/s, Loss: 0.0090]
2026-05-20 16:04:09,122: Epoch 1 average loss: avg_loss: 0.08948, avg_total_loss: 0.08948
# Validation: 100%|██████████| 127/127 [01:43<00:00,  1.23it/s]
2026-05-20 16:05:52,317: Stored metrics at /home/azureuser/NeuralHydrologyAzure/2_final_runs/2_us_127_basins/runs/total_precipitation_sum_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_2005_160237/validation/mod